### Clone repo dan install dependency

In [ ]:
%cd /content
!git clone https://github.com/Comfy-Org/ComfyUI.git
%cd /content/ComfyUI
!pip install -r requirements.txt
!git clone https://github.com/Comfy-Org/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
!pip install -r custom_nodes/ComfyUI-Manager/requirements.txt


### Sambungin GDrive untuk simpan model yg di download dan load model

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')

for folder in ('checkpoints', 'loras', 'vae', 'diffusion_models', 'text_encoders', 'clip'):
    (MODELS_DIR / folder).mkdir(parents=True, exist_ok=True)

# Tambahkan folder Drive ke konfigurasi ComfyUI tanpa menghapus konfigurasi lain.
config = Path('/content/ComfyUI/extra_model_paths.yaml')
existing = config.read_text() if config.exists() else ''

if 'drive_models:' not in existing:
    drive_config = """drive_models:
  base_path: /content/drive/MyDrive/ComfyUI/models
  checkpoints: checkpoints
  loras: loras
  vae: vae
  diffusion_models: diffusion_models
  text_encoders: text_encoders
  clip: clip
"""
    config.write_text(existing.rstrip() + ('\n' if existing.strip() else '') + drive_config)

print('Drive siap:', MODELS_DIR)

### Download VAE, TE dan Diffusion Model ke Drive

In [ ]:
# Unduh VAE, text encoder, dan diffusion model MiniMax H3 ke Google Drive (sekitar 37 GB).
from pathlib import Path
import requests
from tqdm.auto import tqdm

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')
ASSETS = [

    ('vae', 'minimax_h3_audio_vae_fp32.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors'),
    ('vae', 'minimax_h3_video_vae_int8_convrot.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors'),
    ('text_encoders', 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
    ('diffusion_models', 'minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
]

session = requests.Session()
prepared = []
print('Memeriksa tautan dan file yang sudah ada...')
for folder, filename, url in ASSETS:
    destination = MODELS_DIR / folder / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    response = session.head(url, allow_redirects=True, timeout=60)
    response.raise_for_status()
    expected = int(response.headers.get('Content-Length', 0))
    if destination.exists() and expected and destination.stat().st_size == expected:
        print(f'Sudah lengkap, dilewati: {filename}')
        continue
    if destination.exists() and not expected and destination.stat().st_size > 0:
        print(f'File sudah ada (ukuran server tidak tersedia), dilewati: {filename}')
        continue
    prepared.append((destination, url, expected, filename))

if not prepared:
    print('Semua aset sudah tersedia di Drive.')
else:
    total = sum(size for _, _, size, _ in prepared)
    print(f'{len(prepared)} file akan diunduh; total ukuran terdeteksi: {total / (1024**3):.2f} GiB.')
    for destination, url, expected, filename in prepared:
        partial = destination.with_name(destination.name + '.part')
        partial.unlink(missing_ok=True)
        try:
            with session.get(url, stream=True, allow_redirects=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with partial.open('wb') as output, tqdm(
                    total=expected or None, unit='B', unit_scale=True, unit_divisor=1024, desc=filename
                ) as progress:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            output.write(chunk)
                            progress.update(len(chunk))
            downloaded = partial.stat().st_size
            if expected and downloaded != expected:
                raise IOError(f'Ukuran tidak cocok untuk {filename}: {downloaded} dari {expected} bytes.')
            partial.replace(destination)
            print(f'Selesai: {destination}')
        except Exception:
            partial.unlink(missing_ok=True)
            raise

print('Selesai. Jika ComfyUI sudah berjalan, refresh daftar model atau restart server agar model terdeteksi.')

### Download Checkpoint atau lora ke GDrive

In [ ]:
# Isi URL Civitai atau Hugging Face, lalu jalankan cell ini.
URL = 'https://civitai.red/models/827184/wai-illustrious-sdxl?modelVersionId=2883731'
# Untuk file Hugging Face di root repo, pilih folder tujuan di sini.
HF_MODEL_TYPE = 'loras'

from pathlib import Path
from urllib.parse import urlparse, parse_qs
from google.colab import userdata
import requests

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')
parsed = urlparse(URL.strip())
host = (parsed.hostname or '').lower()
if host.startswith('www.'):
    host = host[4:]
parts = parsed.path.strip('/').split('/')
query = parse_qs(parsed.query)

if host == 'huggingface.co':
    if len(parts) < 5 or parts[2] not in ('blob', 'resolve'):
        raise ValueError('Masukkan tautan file Hugging Face dengan format /blob/ atau /resolve/.')
    owner, repo, _, revision = parts[:4]
    file_path = parts[4:]
    if not file_path:
        raise ValueError('Tautan Hugging Face tidak menunjuk ke file.')
    download_url = f'https://huggingface.co/{owner}/{repo}/resolve/{revision}/' + '/'.join(file_path)
    allowed_types = {'checkpoints', 'loras', 'vae', 'diffusion_models', 'text_encoders', 'clip'}
    if file_path[0] in allowed_types:
        kind = file_path[0]
        relative_parts = file_path
    else:
        kind = HF_MODEL_TYPE.strip().lower()
        relative_parts = [file_path[-1]]
    if kind not in allowed_types:
        raise ValueError(f'HF_MODEL_TYPE harus salah satu dari: {sorted(allowed_types)}')
    destination = MODELS_DIR / kind
    for part in relative_parts:
        if part in ('', '.', '..'):
            raise ValueError('Jalur file tidak valid.')
        destination = destination / part
    headers = {}
elif host in ('civitai.com', 'civitai.red'):
    token = None
    try:
        token = userdata.get('CIVITAI_API_TOKEN')
    except Exception:
        pass
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    api_base = f'https://{host}'
    version_id = query.get('modelVersionId', [None])[0]
    file_id = query.get('fileId', [None])[0]
    is_download_link = len(parts) >= 4 and parts[-3:-1] == ['download', 'models']

    if is_download_link:
        model_id = parts[-1]
        response = requests.get(f'{api_base}/api/v1/models/{model_id}', headers=headers, timeout=30)
        response.raise_for_status()
        model_info = response.json()
        chosen = None
        for version in model_info.get('modelVersions') or []:
            for file in version.get('files') or []:
                if file_id and str(file.get('id')) == str(file_id):
                    chosen = (version, file)
                    break
            if chosen:
                break
        if chosen is None:
            version = (model_info.get('modelVersions') or [{}])[0]
            files = version.get('files') or []
            selected_file = next((f for f in files if f.get('primary')), files[0] if files else None)
            if selected_file:
                chosen = (version, selected_file)
        if not chosen:
            raise RuntimeError('Civitai tidak mengembalikan file model.')
        version_info, selected_file = chosen
        download_url = URL.strip()
    else:
        if not version_id:
            raise ValueError('Tautan Civitai perlu memiliki modelVersionId atau berupa /api/download/models/.')
        response = requests.get(f'{api_base}/api/v1/model-versions/{version_id}', headers=headers, timeout=30)
        response.raise_for_status()
        version_info = response.json()
        model_info = version_info.get('model') or {}
        files = version_info.get('files') or []
        selected_file = next((f for f in files if file_id and str(f.get('id')) == str(file_id)), None)
        selected_file = selected_file or next((f for f in files if f.get('primary')), files[0] if files else None)
        if not selected_file:
            raise RuntimeError('Civitai tidak mengembalikan file untuk versi ini.')
        model_id = model_info.get('id')
        download_url = selected_file.get('downloadUrl') or f'{api_base}/api/download/models/{model_id}?fileId={selected_file.get("id")}'

    kind_map = {'lora': 'loras', 'checkpoint': 'checkpoints', 'textualinversion': 'embeddings', 'vae': 'vae', 'controlnet': 'controlnet'}
    kind = kind_map.get((model_info.get('type') or '').lower(), 'loras')
    filename = selected_file.get('name') or 'model.safetensors'
    destination = MODELS_DIR / kind / filename
    headers['Authorization'] = f'Bearer {token}' if token else ''
else:
    raise ValueError('Host belum didukung. Gunakan civitai.com, civitai.red, atau huggingface.co.')

if headers.get('Authorization') == '':
    headers.pop('Authorization')
destination.parent.mkdir(parents=True, exist_ok=True)
if destination.exists():
    print(f'Sudah ada, dilewati: {destination} ({destination.stat().st_size:,} bytes)')
else:
    partial = destination.with_name(destination.name + '.part')
    if partial.exists():
        raise FileExistsError(f'File sementara sudah ada dan tidak akan ditimpa: {partial}')
    print(f'Mengunduh {destination.name} ke {destination}', flush=True)
    try:
        with requests.get(download_url, headers=headers, allow_redirects=True, stream=True, timeout=(60, 180)) as response:
            response.raise_for_status()
            expected = int(response.headers.get('Content-Length', 0)) or None
            downloaded = 0
            with open(partial, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
            if expected is not None and downloaded != expected:
                raise IOError(f'Ukuran unduhan tidak cocok: {downloaded:,} dari {expected:,} bytes.')
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
    print(f'Tersimpan: {destination} ({destination.stat().st_size:,} bytes)')
print('Jika ComfyUI sudah berjalan, refresh daftar model; restart server hanya jika model belum muncul.')

### Jalanin ComfyUI pake cloudflared




In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import time
import socket
import urllib.request

COMFY_DIR = Path("/content/ComfyUI")
COMFY_LOG = Path("/content/comfyui.log")
TUNNEL_LOG = Path("/content/cloudflared.log")
PORT = 8188

def port_is_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            return True
    except OSError:
        return False

# Pasang cloudflared hanya jika runtime baru belum memilikinya.
if shutil.which("cloudflared") is None:
    deb_path = "/tmp/cloudflared-linux-amd64.deb"
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        deb_path,
    )
    subprocess.run(["dpkg", "-i", deb_path], check=True)

if not COMFY_DIR.exists():
    raise FileNotFoundError(f"Folder ComfyUI tidak ditemukan: {COMFY_DIR}")

# Simpan proses sebagai objek latar agar cell segera selesai dan kernel bebas.
comfyui_process = None
if port_is_open(PORT):
    print(f"ComfyUI sudah aktif di port {PORT}; tidak meluncurkan duplikat.", flush=True)
else:
    comfy_log_file = COMFY_LOG.open("w")
    comfyui_process = subprocess.Popen(
        ["python", "main.py", "--dont-print-server"],
        cwd=COMFY_DIR,
        stdout=comfy_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    comfy_log_file.close()
    print(f"ComfyUI diluncurkan di latar (PID {comfyui_process.pid}); menunggu port {PORT}.", flush=True)

    deadline = time.time() + 180
    while time.time() < deadline and not port_is_open(PORT):
        if comfyui_process.poll() is not None:
            tail = COMFY_LOG.read_text(errors="replace")[-5000:]
            raise RuntimeError("ComfyUI berhenti saat startup; lihat log " + str(COMFY_LOG) + " | " + tail)
        time.sleep(2)
    if not port_is_open(PORT):
        raise TimeoutError(f"ComfyUI belum siap setelah 180 detik. Cek {COMFY_LOG}")

# Cegah tunnel duplikat bila cell ini dijalankan ulang di runtime yang sama.
active_tunnel = False
for cmdline in Path("/proc").glob("[0-9]*/cmdline"):
    try:
        args = cmdline.read_bytes().replace(b"\0", b" ").decode(errors="ignore")
        if "cloudflared" in args and "tunnel" in args and f"127.0.0.1:{PORT}" in args:
            active_tunnel = True
            break
    except OSError:
        pass

cloudflared_process = None
if active_tunnel:
    print("Cloudflare Tunnel sudah aktif; gunakan URL yang sudah dicetak sebelumnya.", flush=True)
else:
    tunnel_log_file = TUNNEL_LOG.open("w")
    cloudflared_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=tunnel_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    tunnel_log_file.close()
    print(f"Cloudflare Tunnel diluncurkan di latar (PID {cloudflared_process.pid}).", flush=True)

    deadline = time.time() + 30
    tunnel_url = None
    while time.time() < deadline:
        if cloudflared_process.poll() is not None:
            break
        log_text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            tunnel_url = match.group(0)
            break
        time.sleep(1)
    if tunnel_url:
        print("URL ComfyUI:", tunnel_url, flush=True)
    else:
        print(f"Tunnel masih mulai atau belum memberi URL. Cek log: {TUNNEL_LOG}", flush=True)

print(f"Log ComfyUI: {COMFY_LOG}")
print(f"Log tunnel: {TUNNEL_LOG}")

In [ ]:
from google.colab import output
print(output.eval_js("google.colab.kernel.proxyPort(8188)"))

### Jalanin ComfyUI pake localtunnel




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

### Jalanin ComfyUI pake colab iframe (buat jaga2 klo localtunnel gagal)

Klo liat error 403, coba buka link nya di browser lain

Beberapa fitur UI kayak live image preview kemungkinan ga bekerja karna colab iframe ngeblok websockets.

In [ ]:
import os, socket, subprocess, time
from google.colab import output

log = open('/tmp/comfyui.log', 'w')
server = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--dont-print-server'],
    cwd='/content/ComfyUI', stdout=log, stderr=subprocess.STDOUT, start_new_session=True
)
for _ in range(120):
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        if server.poll() is not None:
            log.flush()
            print(open('/tmp/comfyui.log').read()[-12000:])
            raise RuntimeError('ComfyUI berhenti sebelum port 8188 siap')
        time.sleep(1)
else:
    print(open('/tmp/comfyui.log').read()[-12000:])
    raise TimeoutError('ComfyUI belum membuka port 8188 dalam 120 detik')
output.serve_kernel_port_as_iframe(8188, height=1024)

In [ ]:
import requests
r = requests.get('http://127.0.0.1:8188/', timeout=10)
print('HTTP', r.status_code, 'bytes', len(r.content))
print(r.text[:300])

In [ ]:
import requests
urls = [
  'https://civitai.red/api/download/models/3218603?fileId=3100615',
  'https://civitai.red/api/download/models/2575595?fileId=2463002',
]
for url in urls:
    r = requests.get(url, allow_redirects=True, stream=True, timeout=45)
    print('status:', r.status_code, 'final host:', r.url.split('/')[2])
    print('type:', r.headers.get('Content-Type'), 'size:', r.headers.get('Content-Length'), 'name:', r.headers.get('Content-Disposition'))
    r.close()

In [ ]:
import os, requests
print('DRIVE_MOUNTED:', os.path.ismount('/content/drive'), 'MYDRIVE_EXISTS:', os.path.isdir('/content/drive/MyDrive'))
for version in (3218603, 2575595):
    r = requests.get(f'https://civitai.com/api/v1/model-versions/{version}', timeout=30)
    print('VERSION', version, 'HTTP', r.status_code)
    if r.ok:
        data = r.json()
        print('name:', data.get('name'), '| base:', data.get('baseModel'))
        for f in data.get('files', []):
            print('file:', f.get('name'), '| type:', f.get('type'), '| bytes:', f.get('sizeKB'), 'KB | primary:', f.get('primary'))
    else:
        print(r.text[:300])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('MOUNTED:', __import__('os').path.isdir('/content/drive/MyDrive'))

In [ ]:
from google.colab import userdata
from pathlib import Path
import requests, shutil, os, time
token=userdata.get('CIVITAI_API_TOKEN')
if not token: raise RuntimeError('Secret CIVITAI_API_TOKEN tidak tersedia')
items=[
    (3218603,3100615,'oneObsession_v24.safetensors','checkpoints',6938040682),
    (2575595,2463002,'AIイラストおじさん_V3-Exp-Compact.safetensors','loras',912618100),
]
for mid,fid,name,kind,expected in items:
    local_dir=Path('/content/ComfyUI/models')/kind
    drive_dir=Path('/content/drive/MyDrive/ComfyUI/models')/kind
    local_dir.mkdir(parents=True,exist_ok=True); drive_dir.mkdir(parents=True,exist_ok=True)
    dest=local_dir/name; drive_dest=drive_dir/name
    if dest.exists() or drive_dest.exists(): raise FileExistsError(f'File tujuan sudah ada: {name}; tidak ditimpa')
    part=local_dir/(name+'.part'); drive_part=drive_dir/(name+'.part')
    if part.exists() or drive_part.exists(): raise FileExistsError(f'File .part sudah ada: {name}; tidak ditimpa')
    url=f'https://civitai.com/api/download/models/{mid}?fileId={fid}'
    print(f'Memulai {name} ({expected:,} bytes)',flush=True)
    with requests.get(url,headers={'Authorization':f'Bearer {token}'},allow_redirects=True,stream=True,timeout=(60,180)) as r:
        if r.status_code != 200: raise RuntimeError(f'Unduhan {name} gagal: HTTP {r.status_code}; {r.text[:200]}')
        length=r.headers.get('Content-Length')
        if length and int(length)!=expected: raise RuntimeError(f'Ukuran server tidak sesuai untuk {name}: {length}')
        downloaded=0; report=0; started=time.time()
        with open(part,'wb') as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if chunk:
                    f.write(chunk); downloaded+=len(chunk)
                    if downloaded-report>=256*1024*1024:
                        print(f'  {name}: {downloaded/expected:.0%} ({downloaded:,}/{expected:,} bytes)',flush=True); report=downloaded
            f.flush(); os.fsync(f.fileno())
    if downloaded!=expected: raise RuntimeError(f'Ukuran unduhan tidak cocok untuk {name}: {downloaded:,} dari {expected:,} bytes')
    part.replace(dest)
    print(f'Unduhan lokal selesai: {dest.stat().st_size:,} bytes; menyalin ke Drive...',flush=True)
    shutil.copyfile(dest,drive_part)
    if drive_part.stat().st_size!=expected: raise RuntimeError(f'Ukuran salinan Drive tidak cocok untuk {name}')
    drive_part.replace(drive_dest)
    print(f'TERSIMPAN: {drive_dest} ({drive_dest.stat().st_size:,} bytes)',flush=True)
del token
print('SELESAI: kedua berkas tersimpan di Google Drive.',flush=True)

### Last resort

In [ ]:
import socket
import subprocess
import time
from google.colab import output

def comfyui_aktif():
    try:
        with socket.create_connection(("127.0.0.1", 8188), timeout=2):
            return True
    except OSError:
        return False

if not comfyui_aktif():
    log = open("/content/comfyui.log", "a")
    proses = subprocess.Popen(
        ["python", "main.py", "--dont-print-server"],
        cwd="/content/ComfyUI",
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log.close()

    for _ in range(180):
        if comfyui_aktif():
            break
        time.sleep(1)

if not comfyui_aktif():
    raise RuntimeError("ComfyUI belum aktif. Periksa log di /content/comfyui.log")

url = output.eval_js("google.colab.kernel.proxyPort(8188)")
print("URL ComfyUI:", url)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8188, height=900)